In [ ]:
import sys
import os
from pathlib import Path

# Add parent directory to path if needed
repo_root = Path(__file__).absolute().parents[1] if '__file__' in globals() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

print("Basic imports completed")


In [ ]:
# Install and import GroundingDINO
try:
    from groundingdino.util.inference import load_model, load_image, predict, annotate
    from groundingdino.models import build_model
    from groundingdino.util.slconfig import SLConfig
    from groundingdino.util.utils import clean_state_dict, get_phrases_from_posmap
    import groundingdino.datasets.transforms as T
    print("GroundingDINO imported successfully")
except ImportError:
    print("GroundingDINO not found. Install with: pip install groundingdino-py")
    print("Or clone from: https://github.com/IDEA-Research/GroundingDINO.git")

# Install and import SAM
try:
    from segment_anything import sam_model_registry, SamPredictor
    print("SAM imported successfully")
except ImportError:
    print("SAM not found. Install with: pip install git+https://github.com/facebookresearch/segment-anything.git")

In [ ]:
class GroundedSAM:
    """
    Grounded SAM: Combines GroundingDINO for object detection with SAM for segmentation
    """
    def __init__(
        self,
        grounding_dino_config_path: str = None,
        grounding_dino_checkpoint_path: str = None,
        sam_checkpoint_path: str = None,
        sam_model_type: str = "vit_h",  # "vit_h", "vit_l", "vit_b"
        device: str = "cuda" if torch.cuda.is_available() else "cpu"
    ):
        self.device = device
        
        # Initialize GroundingDINO
        if grounding_dino_config_path and grounding_dino_checkpoint_path:
            try:
                args = SLConfig.fromfile(grounding_dino_config_path)
                args.device = device
                self.grounding_model = build_model(args)
                checkpoint = torch.load(grounding_dino_checkpoint_path, map_location="cpu")
                load_res = self.grounding_model.load_state_dict(
                    clean_state_dict(checkpoint["model"]), strict=False
                )
                self.grounding_model.eval()
                self.grounding_model = self.grounding_model.to(device)
                print(f"GroundingDINO loaded on {device}")
            except Exception as e:
                print(f"Error loading GroundingDINO: {e}")
                self.grounding_model = None
        else:
            print("GroundingDINO config/checkpoint paths not provided")
            self.grounding_model = None
        
        # Initialize SAM
        if sam_checkpoint_path:
            try:
                self.sam = sam_model_registry[sam_model_type](checkpoint=sam_checkpoint_path)
                self.sam.to(device=device)
                self.sam_predictor = SamPredictor(self.sam)
                print(f"SAM ({sam_model_type}) loaded on {device}")
            except Exception as e:
                print(f"Error loading SAM: {e}")
                self.sam_predictor = None
        else:
            print("SAM checkpoint path not provided")
            self.sam_predictor = None
    
    def detect_and_segment(
        self,
        image_path: str,
        text_prompt: str,
        box_threshold: float = 0.3,
        text_threshold: float = 0.25
    ):
        """
        Detect objects using GroundingDINO and segment using SAM
        
        Args:
            image_path: Path to input image
            text_prompt: Text description of objects to detect (e.g., "cat . dog . chair")
            box_threshold: Threshold for box confidence
            text_threshold: Threshold for text similarity
        
        Returns:
            masks: Segmentation masks
            boxes: Bounding boxes
            phrases: Detected object phrases
        """
        if self.grounding_model is None or self.sam_predictor is None:
            raise ValueError("Both GroundingDINO and SAM models must be loaded")
        
        # Load and preprocess image
        image_source, image = load_image(image_path)
        
        # Run GroundingDINO
        boxes, logits, phrases = predict(
            model=self.grounding_model,
            image=image,
            caption=text_prompt,
            box_threshold=box_threshold,
            text_threshold=text_threshold
        )
        
        # Run SAM
        self.sam_predictor.set_image(image_source)
        masks = []
        for box in boxes:
            mask, _, _ = self.sam_predictor.predict(
                point_coords=None,
                point_labels=None,
                box=box[None, :],
                multimask_output=False,
            )
            masks.append(mask[0])
        
        return masks, boxes, phrases
    
    def visualize(self, image_path: str, masks, boxes, phrases):
        """Visualize detection and segmentation results"""
        image_source = cv2.imread(image_path)
        image_source = cv2.cvtColor(image_source, cv2.COLOR_BGR2RGB)
        
        # Annotate image with boxes and labels
        annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=None, phrases=phrases)
        
        # Overlay masks
        for mask in masks:
            annotated_frame = annotated_frame * 0.6 + np.stack([mask*255]*3, axis=2) * 0.4
        
        return annotated_frame.astype(np.uint8)

print("GroundedSAM class defined")


In [ ]:
# Example usage:
# Initialize GroundedSAM (provide paths to model checkpoints)
# grounded_sam = GroundedSAM(
#     grounding_dino_config_path="path/to/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py",
#     grounding_dino_checkpoint_path="path/to/groundingdino_swint_ogc.pth",
#     sam_checkpoint_path="path/to/sam_vit_h_4b8939.pth",
#     sam_model_type="vit_h"
# )

# # Run detection and segmentation
# masks, boxes, phrases = grounded_sam.detect_and_segment(
#     image_path="path/to/image.jpg",
#     text_prompt="cat . dog . chair"
# )

# # Visualize results
# result = grounded_sam.visualize("path/to/image.jpg", masks, boxes, phrases)
# plt.figure(figsize=(10, 10))
# plt.imshow(result)
# plt.axis('off')
# plt.show()

print("Example usage code provided (commented out)")
